# 01 - Exploratory Data Analysis

Loads the raw Brent price series, runs the Extract -> Validate -> Transform pipeline
(`src.data.ingestion`, `src.data.preprocessing`), and inspects stationarity and
distributional properties of the resulting log-return series.

This notebook is intentionally thin: all logic lives in `src/`, tested under `tests/`,
so the notebook is a *view* onto that logic rather than a second copy of it.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis.eda import run_adf_test, summary_statistics
from src.config import PATHS
from src.data.ingestion import load_raw_prices
from src.data.preprocessing import build_processed_prices, save_processed_prices

PATHS.ensure_dirs()

In [ ]:
raw = load_raw_prices()
processed, quality_report = build_processed_prices(raw)
save_processed_prices(processed, PATHS.processed_prices_csv)

print(quality_report.to_dict())
processed.tail()

## Stationarity: raw price vs. log return

Prices are expected to be non-stationary (a random walk with drift); log returns are
expected to be stationary, which is why the change-point model operates on returns.

In [ ]:
price_adf = run_adf_test(processed["Price"], name="Price")
return_adf = run_adf_test(processed["Log_Return"], name="Log_Return")

print("Price:", price_adf.to_dict())
print("Log_Return:", return_adf.to_dict())
assert not price_adf.is_stationary, "expected price level to look non-stationary"
assert return_adf.is_stationary, "expected log returns to be stationary"

In [ ]:
summary_statistics(processed)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(processed["Date"], processed["Price"], linewidth=0.8)
axes[0].set_title("Brent Spot Price (USD/barrel)")
axes[1].plot(processed["Date"], processed["Log_Return"], linewidth=0.5, color="darkorange")
axes[1].set_title("Daily Log Return")
fig.tight_layout()
plt.savefig(PATHS.figures / "eda_price_and_returns.png", dpi=110)
plt.show()

## Next steps

Run `02_change_point_model.ipynb` (or `python pipelines/run_pipeline.py`) to fit the
Bayesian change-point model on `data/processed/brent_log_returns.csv` produced above.